In [ ]:
"""
Market Basket Analysis using the Apriori Algorithm (implemented from scratch)
Mini Project - Data Mining & Analytics
"""
from itertools import combinations

# ---------------------------------------------------------------
# STEP 1: DATASET
# Each row = one customer's basket (a transaction).
# ---------------------------------------------------------------
transactions = [
    {"Bread", "Milk"},
    {"Bread", "Diaper", "Beer", "Eggs"},
    {"Milk", "Diaper", "Beer", "Cola"},
    {"Bread", "Milk", "Diaper", "Beer"},
    {"Bread", "Milk", "Diaper", "Cola"},
    {"Bread", "Milk", "Diaper"},
    {"Milk", "Diaper", "Beer"},
    {"Bread", "Milk", "Eggs"},
    {"Bread", "Diaper", "Beer"},
    {"Milk", "Bread", "Diaper", "Beer"},
]
N = len(transactions)

# ---------------------------------------------------------------
# STEP 2: PARAMETERS
# min_support    : itemset must appear in at least this fraction of baskets
# min_confidence : rule must be correct at least this fraction of the time
# ---------------------------------------------------------------
MIN_SUPPORT = 0.30
MIN_CONFIDENCE = 0.70


def support_count(itemset):
    """Number of transactions that contain every item in `itemset`."""
    return sum(1 for t in transactions if itemset <= t)


def support(itemset):
    return support_count(itemset) / N


print("=" * 60)
print("STEP 1: Dataset -> %d transactions" % N)
for i, t in enumerate(transactions, 1):
    print(f"  T{i}: {sorted(t)}")
print(f"\nSTEP 2: min_support = {MIN_SUPPORT} (>= {MIN_SUPPORT*N:.0f} baskets), "
      f"min_confidence = {MIN_CONFIDENCE}")

# ---------------------------------------------------------------
# STEP 3: FIND FREQUENT 1-ITEMSETS (L1)
# Count every single item, keep those meeting min_support.
# ---------------------------------------------------------------
print("\n" + "=" * 60)
print("STEP 3: Candidate 1-itemsets (C1) and frequent 1-itemsets (L1)")
all_items = sorted({item for t in transactions for item in t})
frequent = {}            # frozenset -> support count (all levels)
Lk = {}
for item in all_items:
    s = frozenset([item])
    c = support_count(s)
    keep = c / N >= MIN_SUPPORT
    print(f"  {item:8s} count={c}  support={c/N:.2f}  -> {'KEEP' if keep else 'PRUNE'}")
    if keep:
        Lk[s] = c
frequent.update(Lk)

# ---------------------------------------------------------------
# STEP 4: ITERATE  (generate Ck -> prune by Apriori property -> count -> Lk)
# Apriori property: every subset of a frequent itemset must be frequent.
# So if any (k-1)-subset of a candidate is NOT frequent, drop it without counting.
# ---------------------------------------------------------------
k = 2
while Lk:
    print("\n" + "=" * 60)
    print(f"STEP 4.{k-1}: Level k={k}")
    prev = list(Lk.keys())

    # (a) JOIN: union pairs of frequent (k-1)-itemsets that give size k
    candidates = set()
    for a, b in combinations(prev, 2):
        u = a | b
        if len(u) == k:
            candidates.add(u)

    # (b) PRUNE using Apriori property
    pruned = set()
    for c in candidates:
        if all(frozenset(sub) in Lk for sub in combinations(c, k - 1)):
            pruned.add(c)
        else:
            print(f"  pruned (infrequent subset) : {sorted(c)}")
    print(f"  Candidates C{k}: {len(candidates)} generated, {len(pruned)} after pruning")

    # (c) COUNT support and keep frequent ones
    newL = {}
    for c in sorted(pruned, key=lambda x: sorted(x)):
        cnt = support_count(c)
        keep = cnt / N >= MIN_SUPPORT
        print(f"  {str(sorted(c)):32s} count={cnt} support={cnt/N:.2f} -> {'KEEP' if keep else 'PRUNE'}")
        if keep:
            newL[c] = cnt
    Lk = newL
    frequent.update(Lk)
    if not Lk:
        print(f"  L{k} is empty -> stop.")
    k += 1

print("\n" + "=" * 60)
print("STEP 5: ALL FREQUENT ITEMSETS")
for s, c in sorted(frequent.items(), key=lambda x: (len(x[0]), -x[1])):
    print(f"  {str(sorted(s)):36s} support={c/N:.2f}")

# ---------------------------------------------------------------
# STEP 6: GENERATE ASSOCIATION RULES
# For each frequent itemset with >=2 items, split into antecedent -> consequent.
#   confidence = supp(A U B) / supp(A)
#   lift       = confidence / supp(B)
# ---------------------------------------------------------------
print("\n" + "=" * 60)
print("STEP 6: Association rules (confidence >= %.2f)" % MIN_CONFIDENCE)
rules = []
for itemset in frequent:
    if len(itemset) < 2:
        continue
    for r in range(1, len(itemset)):
        for A in combinations(itemset, r):
            A = frozenset(A)
            B = itemset - A
            conf = support(itemset) / support(A)
            lift = conf / support(B)
            if conf >= MIN_CONFIDENCE:
                rules.append((A, B, support(itemset), conf, lift))

rules.sort(key=lambda x: (-x[4], -x[3]))
print(f"  {'Rule':38s} {'Supp':>5s} {'Conf':>5s} {'Lift':>5s}")
for A, B, s, c, l in rules:
    print(f"  {str(sorted(A)) + ' -> ' + str(sorted(B)):38s} {s:5.2f} {c:5.2f} {l:5.2f}")

# ---------------------------------------------------------------
# STEP 7: INTERPRET
# ---------------------------------------------------------------
print("\n" + "=" * 60)
print("STEP 7: Interpretation of the top rules")
for A, B, s, c, l in rules[:3]:
    kind = "positively associated" if l > 1 else ("independent" if l == 1 else "negatively associated")
    print(f"  If a customer buys {sorted(A)}, they also buy {sorted(B)} "
          f"{c*100:.0f}% of the time; lift={l:.2f} -> {kind}.")

STEP 1: Dataset -> 10 transactions
  T1: ['Bread', 'Milk']
  T2: ['Beer', 'Bread', 'Diaper', 'Eggs']
  T3: ['Beer', 'Cola', 'Diaper', 'Milk']
  T4: ['Beer', 'Bread', 'Diaper', 'Milk']
  T5: ['Bread', 'Cola', 'Diaper', 'Milk']
  T6: ['Bread', 'Diaper', 'Milk']
  T7: ['Beer', 'Diaper', 'Milk']
  T8: ['Bread', 'Eggs', 'Milk']
  T9: ['Beer', 'Bread', 'Diaper']
  T10: ['Beer', 'Bread', 'Diaper', 'Milk']

STEP 2: min_support = 0.3 (>= 3 baskets), min_confidence = 0.7

STEP 3: Candidate 1-itemsets (C1) and frequent 1-itemsets (L1)
  Beer     count=6  support=0.60  -> KEEP
  Bread    count=8  support=0.80  -> KEEP
  Cola     count=2  support=0.20  -> PRUNE
  Diaper   count=8  support=0.80  -> KEEP
  Eggs     count=2  support=0.20  -> PRUNE
  Milk     count=8  support=0.80  -> KEEP

STEP 4.1: Level k=2
  Candidates C2: 6 generated, 6 after pruning
  ['Beer', 'Bread']                count=4 support=0.40 -> KEEP
  ['Beer', 'Diaper']               count=6 support=0.60 -> KEEP
  ['Beer', 'Milk']   